# Matrix Profiling with STUMPY

This notebook builds a first thesis-ready matrix profile workflow using `stumpy` on the processed BTCUSDT 1-minute dataset.

It is structured to answer four practical questions:

1. Which univariate signal should be profiled first?
2. How can a manageable subset be extracted without losing interpretability?
3. Where are the strongest repeated motifs?
4. Where are the largest discords (unusual subsequences)?

## 1. Setup

If needed, install the project dependencies from the repository root:

```bash
pip install -r requirements.txt
```

In [5]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import stumpy

plt.style.use("seaborn-v0_8-whitegrid")

ModuleNotFoundError: No module named 'stumpy'

In [ ]:
# Resolve paths relative to this notebook location.
project_root = Path("../../").resolve()
data_path = project_root / "data" / "processed" / "crypto" / "1min" / "BTCUSDT_1m_processed.parquet"

data_path

## 2. Load the processed BTCUSDT data

In [ ]:
df = pd.read_parquet(data_path)
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
df = df.sort_values("timestamp").reset_index(drop=True)

print(f"Rows: {len(df):,}")
print(f"Columns: {list(df.columns)}")
print(f"Date range: {df['timestamp'].min()} -> {df['timestamp'].max()}")

## 3. Select a motif-ready signal

For a first matrix profile pass, `log_return` is usually a better starting point than raw price because it reduces trend dominance and makes local pattern comparison more meaningful.

In [ ]:
candidate_column = "log_return"

if candidate_column not in df.columns:
    raise KeyError(f"Expected column '{candidate_column}' was not found.")

signal_df = df[["timestamp", "close", candidate_column]].dropna().reset_index(drop=True)
signal_df = signal_df.rename(columns={candidate_column: "signal"})

print(signal_df.head())
print()
print(signal_df["signal"].describe())

## 4. Define an analysis subset

Running a matrix profile over the full multi-year 1-minute series can be expensive. The cell below keeps the most recent subset so the first experiment remains fast enough to iterate on.

You can increase `subset_size` later if compute time is acceptable.

In [ ]:
subset_size = 30_000
window_size = 60

analysis_df = signal_df.tail(subset_size).copy().reset_index(drop=True)
series = analysis_df["signal"].to_numpy(dtype=float)

if len(series) <= window_size:
    raise ValueError("The subset must be longer than the subsequence window size.")

print(f"Subset length: {len(series):,}")
print(f"Window size: {window_size} observations")
print(f"Profile length will be: {len(series) - window_size + 1:,}")

## 5. Compute the matrix profile

`stumpy.stump` returns one row per subsequence. The first column is the matrix profile value and the second column is the index of the nearest-neighbor subsequence.

In [ ]:
mp = stumpy.stump(series, m=window_size)

profile = np.asarray(mp[:, 0], dtype=float)
neighbor_idx = np.asarray(mp[:, 1], dtype=int)

print(f"Matrix profile shape: {mp.shape}")
print(f"Minimum profile value: {profile.min():.6f}")
print(f"Maximum profile value: {profile.max():.6f}")

In [ ]:
profile_df = pd.DataFrame(
    {
        "subsequence_start": np.arange(len(profile)),
        "timestamp": analysis_df.loc[: len(profile) - 1, "timestamp"].to_numpy(),
        "matrix_profile": profile,
        "nearest_neighbor_start": neighbor_idx,
    }
)

profile_df.head()

## 6. Identify top motifs

Low matrix profile values correspond to repeated patterns. The table below collects the best motif candidates after removing overlapping duplicates.

In [ ]:
def pick_non_overlapping(indices, window, limit):
    selected = []
    for idx in indices:
        if all(abs(idx - chosen) >= window for chosen in selected):
            selected.append(int(idx))
        if len(selected) == limit:
            break
    return selected


motif_indices = np.argsort(profile)
top_motif_indices = pick_non_overlapping(motif_indices, window_size, limit=5)

motif_table = profile_df.loc[top_motif_indices, ["timestamp", "matrix_profile", "nearest_neighbor_start"]].copy()
motif_table = motif_table.sort_values("matrix_profile").reset_index().rename(columns={"index": "subsequence_start"})
motif_table

In [ ]:
best_motif_idx = top_motif_indices[0]
best_neighbor_idx = int(neighbor_idx[best_motif_idx])

motif_a = series[best_motif_idx : best_motif_idx + window_size]
motif_b = series[best_neighbor_idx : best_neighbor_idx + window_size]

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=False)

axes[0].plot(analysis_df["timestamp"], analysis_df["signal"], color="tab:blue", linewidth=0.8)
axes[0].axvspan(
    analysis_df.loc[best_motif_idx, "timestamp"],
    analysis_df.loc[best_motif_idx + window_size - 1, "timestamp"],
    color="tab:green",
    alpha=0.25,
    label="Motif A",
)
axes[0].axvspan(
    analysis_df.loc[best_neighbor_idx, "timestamp"],
    analysis_df.loc[best_neighbor_idx + window_size - 1, "timestamp"],
    color="tab:orange",
    alpha=0.25,
    label="Motif B",
)
axes[0].set_title("Signal with strongest motif pair highlighted")
axes[0].legend()

axes[1].plot(motif_a, label=f"Motif A @ {analysis_df.loc[best_motif_idx, 'timestamp']}", linewidth=2)
axes[1].plot(motif_b, label=f"Motif B @ {analysis_df.loc[best_neighbor_idx, 'timestamp']}", linewidth=2)
axes[1].set_title("Best matching subsequences")
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. Identify discords

Large matrix profile values correspond to subsequences that are unusually dissimilar from the rest of the series.

In [ ]:
discord_indices = np.argsort(profile)[::-1]
top_discord_indices = pick_non_overlapping(discord_indices, window_size, limit=5)

discord_table = profile_df.loc[top_discord_indices, ["timestamp", "matrix_profile", "nearest_neighbor_start"]].copy()
discord_table = discord_table.sort_values("matrix_profile", ascending=False).reset_index().rename(columns={"index": "subsequence_start"})
discord_table

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(analysis_df["timestamp"], analysis_df["signal"], color="tab:blue", linewidth=0.8)
axes[0].set_title("Selected analysis signal")

axes[1].plot(profile_df["timestamp"], profile_df["matrix_profile"], color="tab:red", linewidth=0.8)
axes[1].set_title("Matrix profile")

for idx in top_discord_indices:
    axes[0].axvspan(
        analysis_df.loc[idx, "timestamp"],
        analysis_df.loc[idx + window_size - 1, "timestamp"],
        color="gold",
        alpha=0.25,
    )
    axes[1].axvspan(
        profile_df.loc[idx, "timestamp"],
        profile_df.loc[idx, "timestamp"],
        color="gold",
        alpha=0.4,
    )

plt.tight_layout()
plt.show()

## 8. Interpretation notes

- Motifs with very low profile values indicate repeated local return dynamics.
- Discords mark locally unusual behavior that may correspond to volatility shocks, gaps, or regime transitions.
- For thesis experiments, the next defensible extension is to compare results across multiple window sizes such as 30, 60, 120, and 240 minutes.
- A second extension is to compare BTCUSDT and ETHUSDT on matched calendar windows.